In [1]:
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq
import torch
import torch.nn.functional as F
from tqdm import tqdm

from src.data.dataloader import get_dataloader
from src.models.mm_encoder import MultiModalEncoder
# 加了个
from src.models.timeseries_encoders.base import BaseModel


In [2]:
def retriev_ts_embeddings(args, model, data_loader):
    model.eval()
    all_ts_embeddings = []
    all_text_embeddings = []
    all_timeseries = []
    all_sample_ids = []
    offset = 0

    with torch.no_grad():
        for batch_x in tqdm(data_loader, total=len(data_loader)):
            timeseries = batch_x.timeseries.float().to(args.device)
            input_mask = batch_x.input_mask.long().to(args.device)

            # sample_ids = torch.tensor(batch_x.sample_id, dtype=torch.long).reshape(-1).to(args.device)
            # all_timeseries.append(timeseries)
            # all_ts_embeddings.append(outputs.embeddings)
            # all_text_embeddings.append(outputs.description_emb)
            # all_sample_ids.append(sample_ids)
            
            batch_size = timeseries.size(0)
            sample_ids = torch.arange(offset, offset + batch_size, device=args.device)
            offset += batch_size

            outputs = model(
                x_enc=timeseries,
                input_mask=input_mask,
                channel_description_emb=batch_x.channel_description_emb.to(args.device),
                description_emb=batch_x.description_emb.to(args.device),
                event_emb=batch_x.event_emb.to(args.device),
            )

            #
            all_timeseries.append(timeseries.cpu())
            all_ts_embeddings.append(outputs.embeddings.cpu())
            all_text_embeddings.append(outputs.description_emb.cpu())
            all_sample_ids.append(sample_ids.cpu())

    return {
        "timeseries": torch.cat(all_timeseries, dim=0),
        "ts_embeddings": F.normalize(torch.cat(all_ts_embeddings, dim=0), dim=-1),
        "text_embeddings": F.normalize(torch.cat(all_text_embeddings, dim=0), dim=-1),
        "sample_ids": torch.cat(all_sample_ids, dim=0),
    }


In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


def load_and_embed(checkpoint_path, split="train", batch_size=8):

    # checkpoint = torch.load(checkpoint_path, map_location=device)
    # args = checkpoint["args"]
    
    # model = MultiModalEncoder(args)
    # state_dict = {k.replace("module.", ""): v for k, v in checkpoint["model_state_dict"].items()}
    # model.load_state_dict(state_dict)
    
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    args = checkpoint["args"]

    args.model_name = "TraceEncoder"
    args.rank = 0
    args.world_size = 1
    args.distributed = False
    args.device = device

    ts_state_dict = {}
    for key, value in checkpoint["model_state_dict"].items():
        clean_key = key.replace("module.", "")
        if clean_key.startswith("ts_encoder."):
            ts_state_dict[clean_key[len("ts_encoder."):]] = value

    original_loader = BaseModel.load_pretrained_weights
    BaseModel.load_pretrained_weights = staticmethod(
        lambda *loader_args, **loader_kwargs: {"model_state_dict": ts_state_dict}
    )
    try:
        model = MultiModalEncoder(args)
    finally:
        BaseModel.load_pretrained_weights = original_loader

    state_dict = {
        key.replace("module.", ""): value
        for key, value in checkpoint["model_state_dict"].items()
    }
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()

    args.task_name = "retrieval"
    args.data_split = split
    args.batch_size = batch_size
    args.train_batch_size = batch_size
    args.val_batch_size = batch_size
    args.device = device
    args.distributed = False

    data_loader = get_dataloader(args)
    output = retriev_ts_embeddings(args, model, data_loader)

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return output


In [20]:
checkpoint_path = "results/model_checkpoints/context_align/retriever_demo.pt"
out = load_and_embed(checkpoint_path=checkpoint_path, split="train")
# out = load_and_embed(checkpoint_path=checkpoint_path, split="train", batch_size = 8)

print("ts_embeddings:", tuple(out["ts_embeddings"].shape))
print("text_embeddings:", tuple(out["text_embeddings"].shape))
print("timeseries:", tuple(out["timeseries"].shape))


FileNotFoundError: [Errno 2] No such file or directory: 'results/model_checkpoints/swift-glitter-75\\CATSEncoder.pth'

In [15]:
# Optional: save embeddings for later reuse
output_dir = Path("results/embeddings")
output_dir.mkdir(parents=True, exist_ok=True)

np.save(output_dir / "ts_embedding.npy", out["ts_embeddings"].cpu().numpy())
np.save(output_dir / "text_embedding.npy", out["text_embeddings"].cpu().numpy())
np.save(output_dir / "timeseries.npy", out["timeseries"].cpu().numpy())
np.save(output_dir / "sample_ids.npy", out["sample_ids"].cpu().numpy())

NameError: name 'out' is not defined

In [16]:
parquet_path = Path("dataset/retrieval/train.parquet")
df = pq.read_table(parquet_path).to_pandas()

query_idx = 0  # change this index
query_text_embedding = out["text_embeddings"][query_idx : query_idx + 1]
all_ts_embeddings = out["ts_embeddings"]

mask = torch.arange(all_ts_embeddings.size(0), device=all_ts_embeddings.device) != query_idx
candidate_scores = torch.mm(query_text_embedding, all_ts_embeddings[mask].T).squeeze(0)

topk = 5
topk_scores, topk_pos = torch.topk(candidate_scores, k=topk)
candidate_indices = torch.where(mask)[0][topk_pos]

print("query row index:", int(out["sample_ids"][query_idx]))
print("top-k candidate row indices:", candidate_indices.tolist())


NameError: name 'out' is not defined

In [ ]:
# print("query sample_id:", int(out["sample_ids"][query_idx]))

# query_id = int(out["sample_ids"][query_idx])
# query_row = df[df["id"] == query_id].iloc[0]

query_row = df.iloc[int(out["sample_ids"][query_idx])]
print("\n[Query]")
print("row index:", int(out["sample_ids"][query_idx]))
print("description:", query_row["description"])
print("events:", query_row["events"])

print("\n[Retrieved Top-K]")
for rank, (idx, score) in enumerate(zip(candidate_indices.tolist(), topk_scores.tolist()), start=1):
    row_index = int(out["sample_ids"][idx])
    row = df.iloc[row_index]
    print(f"Top-{rank} | row_index={row_index} | score={score:.4f}")
    print("  description:", row["description"])
    print("  events:", row["events"])
